In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count
import numpy as np
import pandas as pd
import os

spark = SparkSession.builder \
    .appName("Pertemuan6-ParquetETL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


a. EXTRACT

In [4]:
df_transaksi = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
df_produk = spark.read.json("tugas6_produk.json")
df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

print(df_transaksi.count())
print(df_produk.count())
print(df_ulasan.count())

df_transaksi.printSchema()
df_produk.printSchema()
df_ulasan.printSchema()

5000
30
3500
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



B. TRANSFORM — Penggabungan 

In [11]:
df_gabung = df_transaksi.join(df_produk, on="product_id", how="inner")
df_gabung = df_gabung.join(df_ulasan, on="order_id", how="left")
df_gabung = df_gabung.withColumn("total_pendapatan", col("unit_terjual")* col("harga"))

print("Jumlah baris setelah join:", df_gabung.count())
df_gabung.select("order_id", "product_id", "kategori", "unit_terjual", "harga", 
                 "total_pendapatan", "rating").show(10)

Jumlah baris setelah join: 5000
+--------+----------+------------+------------+------+----------------+------+
|order_id|product_id|    kategori|unit_terjual| harga|total_pendapatan|rating|
+--------+----------+------------+------------+------+----------------+------+
|     TX0|        21|  Elektronik|           2| 25000|           50000|     1|
|     TX1|         8|     Fashion|           6|250000|         1500000|  NULL|
|     TX2|        25|  Elektronik|           4| 25000|          100000|     2|
|     TX3|         3|     Fashion|           4| 75000|          300000|     5|
|     TX4|        19|Rumah Tangga|           6| 75000|          450000|  NULL|
|     TX5|        10|  Elektronik|           3| 75000|          225000|     5|
|     TX6|        26|   Kesehatan|           6| 75000|          450000|     4|
|     TX7|         4|     Makanan|           1|150000|          150000|  NULL|
|     TX8|         7|  Elektronik|           2| 75000|          150000|     5|
|     TX9|        30

C. TRANSFORM — Penanganan Data Kosong & Pengayaan 

In [12]:
from pyspark.sql.functions import when

# Menambahkan kolom ada_ulasan
df_etl = df_join.withColumn(
    "ada_ulasan",
    when(col("rating").isNull(), False)
    .otherwise(True)
)

# Mengisi nilai rating yang kosong menjadi 0
df_etl = df_etl.na.fill({
    "rating": 0
})

df_etl.show(10)

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+----------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|ada_ulasan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+----------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|      true|
|         8|     TX1|           6|2026-10-25 00:00:00|     0|250000|     Fashion|   Produk-8|         1500000|     false|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|      true|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|      true|
|        19|     TX4|           6|2026-10-07 00:00:00|     0| 75000|Rumah Tangga|  Produk-19|          450000|     false|
|        10|     TX5|   

D. LOAD 

In [13]:
path = "hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl"

df_etl.write.mode("overwrite") \
    .partitionBy("kategori") \
    .parquet(path)

!hdfs dfs -ls -R /user/mahasiswa/tugas6/hasil_etl

df_final = spark.read.parquet(path)
print(df_final.count())

-rw-r--r--   3 Ziffy2 supergroup          0 2026-09-29 21:10 /user/mahasiswa/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - Ziffy2 supergroup          0 2026-09-29 21:10 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 Ziffy2 supergroup      16363 2026-09-29 21:10 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik/part-00000-8dd83420-6dd2-41d4-838b-9170e9bc174f.c000.snappy.parquet
drwxr-xr-x   - Ziffy2 supergroup          0 2026-09-29 21:10 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 Ziffy2 supergroup      11130 2026-09-29 21:10 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion/part-00000-8dd83420-6dd2-41d4-838b-9170e9bc174f.c000.snappy.parquet
drwxr-xr-x   - Ziffy2 supergroup          0 2026-09-29 21:10 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 Ziffy2 supergroup      10388 2026-09-29 21:10 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan/part-00000-8dd83420-6dd2-41d4-838b-9170e9bc174f.c000.snappy.parquet
drwxr-xr-x   - Zi

E. Insight Akhir 

In [14]:
hasil = (
    df_etl.groupBy("kategori")
    .agg(avg(col("ada_ulasan").cast("double")).alias("persentase_ulasan"))
    .orderBy("persentase_ulasan")
)

hasil.show()

+------------+------------------+
|    kategori| persentase_ulasan|
+------------+------------------+
|     Makanan|0.6884328358208955|
|   Kesehatan|0.6888657648283039|
|     Fashion|0.7014492753623188|
|Rumah Tangga|0.7055214723926381|
|  Elektronik|0.7065940713853599|
+------------+------------------+



F.EXPLOR

In [15]:
#Total Pendapatan per Tanggal
df_etl.groupBy("tanggal") \
      .agg(
          spark_sum("total_pendapatan").alias("pendapatan_harian")
      ) \
      .orderBy("tanggal") \
      .show()

+-------------------+-----------------+
|            tanggal|pendapatan_harian|
+-------------------+-----------------+
|2026-10-01 00:00:00|         78525000|
|2026-10-02 00:00:00|         70125000|
|2026-10-03 00:00:00|         70800000|
|2026-10-04 00:00:00|         74600000|
|2026-10-05 00:00:00|         72075000|
|2026-10-06 00:00:00|         87000000|
|2026-10-07 00:00:00|         69900000|
|2026-10-08 00:00:00|         72525000|
|2026-10-09 00:00:00|         71600000|
|2026-10-10 00:00:00|         74275000|
|2026-10-11 00:00:00|         66525000|
|2026-10-12 00:00:00|         58225000|
|2026-10-13 00:00:00|         65500000|
|2026-10-14 00:00:00|         66025000|
|2026-10-15 00:00:00|         80950000|
|2026-10-16 00:00:00|         72450000|
|2026-10-17 00:00:00|         67300000|
|2026-10-18 00:00:00|         68975000|
|2026-10-19 00:00:00|         69025000|
|2026-10-20 00:00:00|         64800000|
+-------------------+-----------------+
only showing top 20 rows



In [16]:
#Produk Paling Sering Terjual
df_etl.groupBy("nama_produk") \
      .agg(
          spark_sum("unit_terjual").alias("total_unit")
      ) \
      .orderBy("total_unit", ascending=False) \
      .show(10)

+-----------+----------+
|nama_produk|total_unit|
+-----------+----------+
|  Produk-12|       788|
|  Produk-17|       757|
|   Produk-7|       753|
|  Produk-10|       742|
|  Produk-18|       731|
|  Produk-13|       715|
|   Produk-4|       710|
|  Produk-21|       709|
|  Produk-26|       702|
|  Produk-20|       696|
+-----------+----------+
only showing top 10 rows



In [17]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
